# Week 2: Basic Stress-Strain Analysis with AI Assistance

## Comments

Notice how the entire lesson is written in Markdown cells. This means to use any of the code you will need to highlight it and paste it into a code cell and then execute (Shift+Enter). 

## Data Loading and Validation

### 1. Sample Dataset
Let's read in some data: `Al7075.csv`
- we need to get an idea of what our data looks like by printing out the first few rows and the last few rows
- we are looking to see if any metadata has been included in the file
- if there are any blank rows at the beginning
- where the column headers are located

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

file_path='Al7075.csv' # if the data file is in the same location as your notebook
data=pd.read_csv(file_path)

print("First 5 rows")
print(data[:5])
print("")
print("Last 5 rows")
print(data[-5:])

```

#### output>
```
First 5 rows
  radius          0.25           in   Unnamed: 3    Unnamed: 4
0    NaN           NaN          NaN          NaN           NaN
1  Point  Length_clean    Force_lbs    Length_in         noise
2      0             2            0            2             0
3      1        2.0033  3189.705354  2.001917956  -0.344941754
4      2       2.00644  6322.451684  2.004027394  -0.601215646

Last 5 rows
   radius     0.25           in   Unnamed: 3    Unnamed: 4
17     15  2.19034  16973.78921  2.186209794  -0.942823067
18     16   2.2194  17002.26872  2.216227176  -0.714793105
19     17     2.25  17002.26872  2.250524887   0.116641612
20     18    2.278  16831.39164  2.274959584  -0.667343174
21     19   2.3066   16261.8014   2.30634202  -0.055922108
```

From this we see: 
- row 0 has metadata: radius, 0.25, in
- row 1 has no data: NaN
- row 2 has headers: `['Point', 'Length_clean', 'Force_lbs', 'Length_in', 'noise']`
- row 3 starts our data

### Now we can read in our data

#### Use the data in "Force_lbs" and "Length_in"

```python
def load_and_analyze_tensile_data(file_path):
    """Load and analyze tensile test data efficiently"""
    
    # Load metadata (first line)
    metadata = pd.read_csv(file_path, header=None, nrows=1)
    radius_inches = metadata.iloc[0, 1]
    radius_units = metadata.iloc[0, 2]
    
    # Load data (starting from row 3)
    data = pd.read_csv(file_path, header=2)
    
    print("=== DATA LOADED SUCCESSFULLY ===")
    print(f"Sample radius: {radius_inches} {radius_units}")
    print(f"Dataset shape: {data.shape}")
    print(f"Columns: {data.columns.tolist()}")
    print(f"\nFirst few rows:")
    print(data.head())
    
    return data, metadata

# Load the data
data, metadata = load_and_analyze_tensile_data("place your file path here")
```

#### output>
```
=== DATA LOADED SUCCESSFULLY ===
Sample radius: 0.25 in
Dataset shape: (20, 5)
Columns: ['Point', 'Length_clean', 'Force_lbs', 'Length_in', 'noise']

First few rows:
   Point  Length_clean     Force_lbs  Length_in     noise
0      0       2.00000      0.000000   2.000000  0.000000
1      1       2.00330   3189.705354   2.001918 -0.344942
2      2       2.00644   6322.451684   2.004027 -0.601216
3      3       2.00948   9284.320941   2.011715  0.556187
4      4       2.01210  11818.997520   2.011288 -0.201879
```

### Next in our workflow is to check the data quality

#### 2. Data Quality Check
```python
def check_data_quality(data):
    """Enhanced data quality assessment with actionable feedback"""
    print("=== DATA QUALITY CHECK ===")
    
    issues_found = [] #initializing variable to hold issues found
    
    # Check for missing values
    missing_count = data.isnull().sum().sum()
    if missing_count > 0:
        issues_found.append(f"Missing values: {missing_count}")
        print(f"⚠️  Found {missing_count} missing values")
        print("   Recommendation: Consider data.dropna() or interpolation")
    
    # Check for negative load values
    negative_load = (data['Force_lbs'] < 0).sum()
    if negative_load > 0:
        issues_found.append(f"Negative load values: {negative_load}")
        print(f"⚠️  Found {negative_load} negative load values")
        print("   Recommendation: Filter with data[data['Force_lbs'] >= 0]")
    
    # Summary
    if issues_found:
        print(f"\n�� Issues found: {len(issues_found)}")
        print("   Address these before proceeding with analysis")
    else:
        print("✅ Data quality check passed!")
    
    return data, issues_found

# Run quality check
data,issues = check_data_quality(data)
if issues:
    print(issues)
else: print("No issues")
```

### Basic Stress-Strain Analysis

#### 1. Calculate Engineering Properties
```python
def stress_strain_from_load_displacement(data, lbs_to_N_factor, radius_mm):
    stress=lbs_to_N_factor*data['Force_lbs']/(np.pi*radius_mm**2) #units MPa
    strain=(data['Length_in']-data['Length_in'][0])/data['Length_in'][0] #unitless
    data['Stress_MPa']=stress
    data['Strain']=strain
    return data

def calculate_mechanical_properties(data):
    """Calculate basic mechanical properties"""
    stress = data['Stress_MPa'].values
    strain = data['Strain'].values
    
    # Young's Modulus (slope of linear region)
    # Use first 7 points for linear region
    linear_end = 7
    slope, intercept = np.polyfit(strain[:linear_end], stress[:linear_end], 1)
    E = slope
    
    # Yield Strength (0.2% offset method)
    offset_strain = strain + 0.002
    offset_stress = E * offset_strain
    yield_idx = np.argmin(np.abs(stress - offset_stress))
    sigma_y = stress[yield_idx]
    
    # Ultimate Tensile Strength
    sigma_u = np.max(stress)
    
    # Elongation at break
    elongation = strain[-1] * 100
    
    return {
        'Young_Modulus_MPa': E,
        'Yield_Strength_MPa': sigma_y,
        'Ultimate_Strength_MPa': sigma_u,
        'Elongation_Percent': elongation
    }

# Calculate stress - strain
radius=metadata.iloc[0,1] * 25.4 #convert inches to mm
data=stress_strain_from_load_displacement(data, 4.448, radius)

# Calculate properties
properties = calculate_mechanical_properties(data)
print("\n=== MECHANICAL PROPERTIES ===")
for prop, value in properties.items():
    print(f"{prop}: {value:.1f}")
```

#### 2. Create Basic Visualizations
```python
def create_stress_strain_plot(data, properties):
    """Create publication-ready stress-strain plot"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Main stress-strain curve
    ax1.plot(data['Strain'], data['Stress_MPa'], 'b.', linewidth=2, label='Experimental Data')
    
    # Add linear region line
    linear_end = int(len(data) * 0.55)
    strain_linear = data['Strain'].iloc[:linear_end]
    stress_linear = properties['Young_Modulus_MPa'] * strain_linear
    ax1.plot(strain_linear, stress_linear, 'r--', linewidth=2, label=f"E = {properties['Young_Modulus_MPa']:.0f} MPa")

     # Add linear region line
    linear_end = int(len(data) * 0.55)
    strain_linear = data['Strain'].iloc[:linear_end]
    stress_linear = properties['Young_Modulus_MPa'] * strain_linear
    ax2.plot(strain_linear, stress_linear, 'r--', linewidth=2, label=f"E = {properties['Young_Modulus_MPa']:.0f} MPa")

    # Add 0.002 offset line
    linear_end = int(len(data) * 0.55)
    strain_linear1 = (data['Strain'].iloc[:linear_end])+0.002
    stress_linear1 = properties['Young_Modulus_MPa'] * (strain_linear1-0.002)
    ax2.plot(strain_linear1, stress_linear1, 'k--', linewidth=2, label=f"E = {properties['Young_Modulus_MPa']:.0f} MPa")
    
    # Add yield point
    ax1.axhline(y=properties['Yield_Strength_MPa'], color='g', linestyle=':', 
                label=f'σy = {properties["Yield_Strength_MPa"]:.0f} MPa')
    
    ax1.set_xlabel('Engineering Strain')
    ax1.set_ylabel('Engineering Stress (MPa)')
    ax1.set_title('Aluminum 7075-T6 Stress-Strain Curve')
    ax1.set_xlim(0, 0.16)
    ax1.set_ylim(0, 700)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Zoom on elastic region
    ax2.plot(data['Strain'], data['Stress_MPa'], 'bo', linewidth=2)
    ax2.plot(strain_linear, stress_linear, 'r--', linewidth=2)
    ax2.set_xlim(0, 0.016)
    ax2.set_ylim(0, 600)
    ax2.set_xlabel('Engineering Strain')
    ax2.set_ylabel('Engineering Stress (MPa)')
    ax2.set_title('Elastic Region (Zoom)')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Create plots
create_stress_strain_plot(data, properties)
```

### Interpretation

Looking at the calculated values (shown below) and at the plots above, are there any problems with our code? Consider: 
- Do all calculated mechanical properties seem consistent with the data set?
- Do all values seem reasonable compared to known values for this alloy?

Are there any problems with our data set? 

```python
=== MECHANICAL PROPERTIES ===
Young_Modulus_MPa: 58253.8
Yield_Strength_MPa: 222.0
Ultimate_Strength_MPa: 597.0
Elongation_Percent: 15.3
```

### AI-Assisted Analysis Enhancement

#### Using AI for Interpretation
Now let's use our AI tools to enhance our analysis:

**Prompt for AI**:
```
I've analyzed aluminum 7075-T6 tensile test data and found:
- Young's Modulus: [enter value from above] MPa
- Yield Strength: [enter value from above] MPa  
- Ultimate Strength: [enter value from above] MPa
- Elongation: [enter value from above]%

The data shows some noise in the stress measurements. Can you help me:
1. Interpret these results compared to typical Al 7075-T6 properties?
2. Suggest ways to reduce noise in future measurements?
3. Identify any potential issues with my analysis method?
4. Recommend additional properties I should calculate?
```

#### AI Response Analysis
The AI should provide:
- Comparison with literature values
- Noise reduction strategies
- Analysis validation suggestions
- Additional property calculations

## Week 2 Assignment: Complete Stress-Strain Analysis

**Due**: End of Week 2  
**Points**: 15 points  
**Deliverables**:
1. **Complete but clean analysis code** with all mechanical property calculations
2. **Publication-ready stress-strain plot** with proper labels and formatting
3. **AI interaction summary** showing how AI enhanced your analysis
4. **Results validation** comparing your values with literature
5. **Error analysis** identifying potential sources of uncertainty

**Code Requirements**:
- Clean, well-documented functions
- Error handling
- Clear variable naming
- Comprehensive comments throughout

**Analysis Requirements**:
- Calculate Young's modulus, yield strength, ultimate strength, elongation
- Create professional stress-strain plots
- Validate results against literature values
- Document any data quality issues
- Reflection

**Reflections to Include**:
- Outline/Summarize your analysis process and results.
- Written comments and interpretation should be in your own words. Include only comments relevant to this problem and its calculations. This is where you need to interpret the information from the AI output to summarize only the relevant points and/or add your own points.
- Remember to include a comment section on how you used AI to help and how helpful or not helpful it was. Reflect on how you might change your initial prompt i.e. what did you learn through your interaction?

## Key Concepts Summary

### AI Tool Integration
- **ChatGPT/Claude**: Research planning and interpretation assistance

### Data Analysis Workflow
1. **Data Loading**: Import and validate data quality
2. **Property Calculation**: Implement mechanical property algorithms
3. **Visualization**: Create publication-ready plots
4. **AI Enhancement**: Use AI for interpretation and validation
5. **Documentation**: Record analysis process and results

### Best Practices
- **Always validate AI suggestions** with domain knowledge
- **Test code incrementally** to catch errors early
- **Document your analysis process** for reproducibility
- **Compare results with literature** to validate findings
- **Consider data quality issues** before drawing conclusions

---

## Next Steps

In the next lesson, we'll learn **prompt engineering** techniques to get better results from AI tools, and apply them to a more complex **alloy optimization case study**.

**Remember**: AI tools are assistants, not replacements for your materials science expertise. Use them to enhance your analysis, not to replace critical thinking.

---

## Resources and References

### AI Tools
- [ChatGPT Plus](https://chat.openai.com)
- [Claude Pro](https://claude.ai)
- [GitHub Copilot](https://github.com/features/copilot)


### Python Resources
- [Pandas Documentation](https://pandas.pydata.org/docs/)
- [Matplotlib Tutorial](https://matplotlib.org/stable/tutorials/index.html)
- [NumPy User Guide](https://numpy.org/doc/stable/user/index.html)

---

**Good luck with your first AI-augmented materials science analysis!** 🚀